# LeetCode 3452 — Sum of Good Numbers bằng Python

Link đề: https://leetcode.com/problems/sum-of-good-numbers/ · Độ khó: **Easy** · Chủ đề: Array (mảng), duyệt theo chỉ số, kiểm tra biên

Skeleton LeetCode:

```python
class Solution:
    def sumOfGoodNumbers(self, nums: List[int], k: int) -> int:
        ...
```

Ghi chú: hai ví dụ chính lấy từ đề; mọi đáp án trong notebook đã được kiểm chứng bằng code (đối chiếu năm cách làm trên hàng nghìn mảng ngẫu nhiên). Ví dụ nào ghi "tự dựng" là do mình thêm để minh họa — nếu cần khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một phần tử `nums[i]` là **tốt (good)** nếu nó **lớn hơn hẳn (strictly greater)** phần tử ở chỉ số `i - k` **và** phần tử ở chỉ số `i + k` (chỉ so với những chỉ số **tồn tại**; nếu cả hai chỉ số đều không tồn tại thì `nums[i]` vẫn tốt). Cần trả về **tổng** các phần tử tốt.
2. Nắm các công cụ Python: `enumerate`, **kiểm tra biên chỉ số** (`i - k >= 0`, `i + k < n`), **chỉ số âm** trong list (`nums[-1]` là phần tử cuối — bẫy lớn!), toán tử `and`/`or` **đoản mạch**, `continue`, `sum` với generator, và **phần tử lính canh (sentinel)** / **đệm (padding)**.
3. Đi từ cách **cờ `good` kiểm tra từng phía** → **hàm tra phần tử trả `0` khi ra ngoài mảng** → **`continue` loại các ca không tốt** → **một biểu thức `sum(...)`** → **đệm hai đầu bằng `0`**.
4. Hiểu mấu chốt: với mỗi vị trí `i` chỉ cần so với **tối đa hai** phần tử, nên một lượt duyệt `O(n)` là đủ; điều khó nhất là xử lý đúng **biên** (không để chỉ số âm hay vượt mảng gây sai/lỗi).
5. Tránh các lỗi kinh điển: dùng `nums[i - k]` khi `i < k` (Python **không báo lỗi** mà lấy phần tử cuối mảng), nhầm `>` với `>=`, so với `i - 1` / `i + 1` thay vì `i - k` / `i + k`, kiểm tra biên sai (`i + k <= n`), và cộng **số lượng** thay vì **tổng**.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và một số nguyên `k`. Phần tử `nums[i]` được gọi là **tốt (good)** nếu nó **lớn hơn hẳn (strictly)** các phần tử ở chỉ số `i - k` và `i + k` (nếu các chỉ số đó tồn tại). Nếu **không** chỉ số nào trong hai chỉ số đó tồn tại thì `nums[i]` vẫn được coi là tốt.

Hãy trả về **tổng** của tất cả các phần tử tốt trong mảng.

**Ví dụ 1**

- Input: `nums = [1,3,2,1,5,4]`, `k = 2`
- Output: `12`
- Giải thích: các số tốt là `nums[1] = 3`, `nums[4] = 5` và `nums[5] = 4`, vì mỗi số lớn hơn hẳn các số ở chỉ số `i - k` và `i + k` (nếu có). Tổng `3 + 5 + 4 = 12`.

**Ví dụ 2**

- Input: `nums = [2,1]`, `k = 1`
- Output: `2`
- Giải thích: số tốt duy nhất là `nums[0] = 2` vì nó lớn hơn hẳn `nums[1]` (chỉ số `0 - 1 = -1` không tồn tại). `nums[1] = 1` không tốt vì `1 < nums[0] = 2`.

**Ràng buộc:** `2 <= nums.length <= 100`; `1 <= nums[i] <= 1000`; `1 <= k <= floor(nums.length / 2)`.

In [1]:
import random
from typing import List

examples = [
    ([1, 3, 2, 1, 5, 4], 2, 12),
    ([2, 1], 1, 2),
]
for nums, k, expected in examples:
    print(nums, "k =", k, "-> kỳ vọng", expected)

[1, 3, 2, 1, 5, 4] k = 2 -> kỳ vọng 12
[2, 1] k = 1 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Chỉ số, `len` và `enumerate`

Phần tử của list đánh chỉ số từ `0` đến `n - 1` (với `n = len(nums)`). `enumerate(nums)` cho ta **cả chỉ số `i` lẫn giá trị `x`** trong mỗi vòng lặp: `for i, x in enumerate(nums):`.

In [2]:
nums = [1, 3, 2, 1, 5, 4]
n = len(nums)
print("n =", n, "| chỉ số hợp lệ: 0 ..", n - 1)
for i, x in enumerate(nums):
    print(f"i={i}, nums[i]={x}")

n = 6 | chỉ số hợp lệ: 0 .. 5
i=0, nums[i]=1
i=1, nums[i]=3
i=2, nums[i]=2
i=3, nums[i]=1
i=4, nums[i]=5
i=5, nums[i]=4


### 3.2. Kiểm tra biên: chỉ số có tồn tại không?

Với phần tử ở `i`, hai "hàng xóm xa" là `i - k` (bên trái) và `i + k` (bên phải).

- `i - k` **tồn tại** khi `i - k >= 0`, tức `i >= k`.
- `i + k` **tồn tại** khi `i + k < n` (chỉ số lớn nhất là `n - 1`).

Ví dụ `n = 6`, `k = 2`: `i = 1` có `i - k = -1` (không tồn tại), `i + k = 3` (tồn tại).

In [3]:
n, k = 6, 2
for i in range(n):
    left = i - k if i - k >= 0 else None
    right = i + k if i + k < n else None
    print(f"i={i}: trái={left}, phải={right}")

i=0: trái=None, phải=2
i=1: trái=None, phải=3
i=2: trái=0, phải=4
i=3: trái=1, phải=5
i=4: trái=2, phải=None
i=5: trái=3, phải=None


### 3.3. Chỉ số **âm** trong Python — bẫy lớn

Trong Python, `nums[-1]` là phần tử **cuối**, `nums[-2]` là phần tử áp cuối... Vì vậy nếu bạn viết `nums[i - k]` khi `i - k < 0`, Python **không báo lỗi** mà lặng lẽ lấy phần tử từ **phía cuối** mảng — kết quả sai mà không có cảnh báo. Ngược lại `nums[i + k]` với `i + k >= n` thì báo `IndexError`. Luôn kiểm tra biên **trước** khi truy cập.

In [4]:
nums = [1, 3, 2, 1, 5, 4]
print(nums[-1], nums[-2])             # 4 5   (cuối mảng)
i, k = 1, 2
print("i - k =", i - k, "-> nums[i - k] =", nums[i - k])    # nums[-1] = 4: KHÔNG phải 'không tồn tại'!
try:
    nums[10]
except IndexError as e:
    print("IndexError:", e)

4 5
i - k = -1 -> nums[i - k] = 4
IndexError: list index out of range


### 3.4. `and` / `or` đoản mạch và `continue`

`A and B`: nếu `A` sai thì Python **không tính** `B`. Nhờ đó `i >= k and x <= nums[i - k]` an toàn: khi `i < k` thì vế sau **không bao giờ chạy**, không bị lấy nhầm chỉ số âm. Tương tự `i + k < n and x <= nums[i + k]`.

`continue` bỏ qua phần còn lại của vòng lặp hiện tại, chuyển sang phần tử kế tiếp.

In [5]:
nums = [1, 3, 2, 1, 5, 4]
i, k = 1, 2
print(i >= k and nums[i] <= nums[i - k])       # False: vế phải KHÔNG chạy vì i < k
print(i + k < len(nums) and nums[i] <= nums[i + k])   # False: 3 <= 1 sai

for x in range(5):
    if x % 2 == 0:
        continue                                # bỏ qua số chẵn
    print("lẻ:", x)

False
False
lẻ: 1
lẻ: 3


### 3.5. So sánh **lớn hơn hẳn** (`>`) và `sum` với generator

"Strictly greater" nghĩa là dùng `>` (không phải `>=`): hai số **bằng nhau** thì **không** tốt. Điều kiện **không tốt** là `x <= hàng_xóm` (kể cả bằng).

`sum(x for ... if ...)` cộng các `x` thỏa điều kiện mà không cần list trung gian (generator expression).

In [6]:
print(3 > 3, 3 >= 3)                              # False True
print(sum(x for x in [1, 2, 3, 4] if x % 2 == 0)) # 2 + 4 = 6
print(sum(x for x in [1, 3, 5] if x > 10))        # 0 (không có phần tử nào) - sum rỗng = 0

False True
6
0


### 3.6. Lính canh (sentinel) và đệm (padding)

Thay vì kiểm tra biên ở mọi chỗ, ta có thể **đệm thêm** giá trị giả ở hai đầu để các chỉ số "ra ngoài" trở thành chỉ số **hợp lệ** nhưng chứa giá trị **không bao giờ cản đường**. Ở đây mọi `nums[i] >= 1`, nên giá trị `0` đệm ở hai đầu luôn **nhỏ hơn** mọi `nums[i]` ⇒ phép so "lớn hơn hẳn" luôn đúng với phần tử giả. Nối list: `[0] * k + nums + [0] * k`.

In [7]:
nums = [1, 3, 2, 1, 5, 4]
k = 2
padded = [0] * k + nums + [0] * k
print(padded)                          # [0, 0, 1, 3, 2, 1, 5, 4, 0, 0]
print("nums[i] nằm ở padded[i + k]; hàng xóm: padded[i], padded[i + 2 * k]")
i = 1
print(padded[i + k], padded[i], padded[i + 2 * k])    # 3 0 1  -> 3 > 0 và 3 > 1

[0, 0, 1, 3, 2, 1, 5, 4, 0, 0]
nums[i] nằm ở padded[i + k]; hàng xóm: padded[i], padded[i + 2 * k]
3 0 1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `nums = [1, 3, 2, 1, 5, 4]` và `k = 2` (`n = 6`), xét từng vị trí:

| `i` | `nums[i]` | trái `i-k` | phải `i+k` | điều kiện | tốt? |
|---|---|---|---|---|---|
| 0 | 1 | không có | `nums[2] = 2` | `1 > 2`? Sai | không |
| 1 | 3 | không có | `nums[3] = 1` | `3 > 1`? Đúng | **có** |
| 2 | 2 | `nums[0] = 1` | `nums[4] = 5` | `2 > 1` đúng, `2 > 5` sai | không |
| 3 | 1 | `nums[1] = 3` | `nums[5] = 4` | `1 > 3`? Sai | không |
| 4 | 5 | `nums[2] = 2` | không có (`6 >= n`) | `5 > 2` đúng | **có** |
| 5 | 4 | `nums[3] = 1` | không có (`7 >= n`) | `4 > 1` đúng | **có** |

Lưu ý dòng `i = 3`: `i + k = 5 < 6` nên **có** hàng xóm phải `nums[5] = 4`, và hàng xóm trái là `nums[1] = 3`. Chỉ cần `1 > 3` sai là đã bị loại (không cần xét tiếp).

Tổng các số tốt: `3 + 5 + 4 = 12`. Quy tắc nhớ: **phần tử tốt = thắng MỌI hàng xóm xa tồn tại**; chỉ cần thua hoặc hòa **một** hàng xóm là bị loại.

In [8]:
nums, k = [1, 3, 2, 1, 5, 4], 2
n = len(nums)
total = 0
for i, x in enumerate(nums):
    left = nums[i - k] if i - k >= 0 else None
    right = nums[i + k] if i + k < n else None
    good = (left is None or x > left) and (right is None or x > right)
    print(f"i={i} x={x} trái={left} phải={right} -> {'TỐT' if good else 'không'}")
    if good:
        total += x
print("tổng =", total)

i=0 x=1 trái=None phải=2 -> không
i=1 x=3 trái=None phải=1 -> TỐT
i=2 x=2 trái=1 phải=5 -> không
i=3 x=1 trái=3 phải=4 -> không
i=4 x=5 trái=2 phải=None -> TỐT
i=5 x=4 trái=1 phải=None -> TỐT
tổng = 12


## 5. Cách 1 — Brute Force / baseline: cờ `good` kiểm tra từng phía

**Ý tưởng.** Bám sát đề: với mỗi `i`, đặt `good = True`. Nếu hàng xóm **trái** tồn tại (`i - k >= 0`) mà `nums[i] <= nums[i - k]` thì `good = False`. Tương tự với hàng xóm **phải** (`i + k < n`). Sau hai kiểm tra, nếu `good` còn `True` thì cộng `nums[i]` vào tổng.

Đây là cách dài dòng nhưng **dễ kiểm tra từng điều kiện** nhất.

In [9]:
def sum_good_flag(nums: List[int], k: int) -> int:
    n = len(nums)
    total = 0
    for i in range(n):
        good = True
        if i - k >= 0 and nums[i] <= nums[i - k]:      # hàng xóm trái tồn tại và không nhỏ hơn
            good = False
        if i + k < n and nums[i] <= nums[i + k]:       # hàng xóm phải tồn tại và không nhỏ hơn
            good = False
        if good:
            total += nums[i]
    return total


for nums, k, expected in examples:
    print(sum_good_flag(nums, k), "(kỳ vọng", expected, ")")

12 (kỳ vọng 12 )
2 (kỳ vọng 2 )


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)`. Mỗi phần tử so với tối đa hai hàng xóm ⇒ **Time `O(n)`**, **Space `O(1)`**. Cách này **đã tối ưu về bậc**; các cách sau chỉ khác về **cách viết** (ngắn gọn hơn, ít lỗi biên hơn).

**Điểm vướng:** hai khối `if` lặp cấu trúc và dễ nhầm biên. Ta muốn viết gọn mà vẫn an toàn.

## 6. Ý tưởng thuật toán / bước đột phá

Không có "bước đột phá về bậc" ở bài này (đã `O(n)`), nhưng có **một ý tưởng làm code sạch hơn**: coi các chỉ số **ra ngoài mảng** như có giá trị **rất nhỏ** (không bao giờ cản `nums[i]`). Ba cách thể hiện:

1. **Lính canh:** hàm `value_at(i)` trả `0` khi `i` ra ngoài (an toàn vì `nums[i] >= 1`).
2. **Điều kiện loại sớm:** `if (có hàng xóm trái và x <= nó) or (có hàng xóm phải và x <= nó): continue`.
3. **Đệm mảng:** thêm `k` số `0` ở mỗi đầu để mọi chỉ số đều hợp lệ.

Ví dụ ý tưởng "ra ngoài = 0": với `nums = [2, 1]`, `k = 1`, `i = 0`: hàng xóm trái là `0` (ngoài mảng), phải là `1`; `2 > 0` và `2 > 1` ⇒ tốt.

In [10]:
nums, k = [2, 1], 1
n = len(nums)


def value_at(i):
    return nums[i] if 0 <= i < n else 0       # ra ngoài mảng -> 0

for i, x in enumerate(nums):
    print(f"i={i} x={x}: trái={value_at(i - k)}, phải={value_at(i + k)}, tốt = {x > value_at(i - k) and x > value_at(i + k)}")

i=0 x=2: trái=0, phải=1, tốt = True
i=1 x=1: trái=2, phải=0, tốt = False


## 7. Các cách tiếp theo

### Cách 2 — Hàm tra phần tử trả `0` khi ra ngoài mảng (lính canh)

**Giải thích từng đoạn.**

1. `value_at(j)`: nếu `0 <= j < n` trả `nums[j]`, ngược lại trả `0`. Biểu thức điều kiện `a if cond else b` gọn.
2. Phần tử `nums[i]` tốt khi `nums[i] > value_at(i - k)` **và** `nums[i] > value_at(i + k)`.
3. Vì `nums[i] >= 1 > 0`, giá trị lính canh `0` **không bao giờ** làm một phần tử tốt bị loại oan. (Nếu đề cho số `<= 0`, phải dùng `-inf`.)

In [11]:
def sum_good_sentinel(nums: List[int], k: int) -> int:
    n = len(nums)

    def value_at(j: int) -> int:
        return nums[j] if 0 <= j < n else 0        # lính canh: ngoài mảng coi như 0

    total = 0
    for i, x in enumerate(nums):
        if x > value_at(i - k) and x > value_at(i + k):
            total += x
    return total


for nums, k, expected in examples:
    print(sum_good_sentinel(nums, k), "(kỳ vọng", expected, ")")

12 (kỳ vọng 12 )
2 (kỳ vọng 2 )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionSentinel:
    def sumOfGoodNumbers(self, nums: List[int], k: int) -> int:
        n = len(nums)

        def value_at(j: int) -> int:
            return nums[j] if 0 <= j < n else 0

        return sum(x for i, x in enumerate(nums) if x > value_at(i - k) and x > value_at(i + k))


print(SolutionSentinel().sumOfGoodNumbers([1, 3, 2, 1, 5, 4], 2))      # 12

12


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(1)`** (hàm lồng nhau không dùng thêm mảng).

### Cách 3 — Loại sớm bằng `continue` (cách mình khuyên nộp)

**Giải thích từng đoạn.**

1. Duyệt `i, x` bằng `enumerate`.
2. Nếu hàng xóm trái **tồn tại** (`i >= k`) và `x <= nums[i - k]`: `x` **không tốt** ⇒ `continue`.
3. Nếu hàng xóm phải **tồn tại** (`i + k < n`) và `x <= nums[i + k]`: `continue`.
4. Qua hết hai "cửa ải" thì `x` tốt: `ans += x`.

Cách này **đọc đúng như đề** ("bị loại nếu không lớn hơn hẳn một hàng xóm tồn tại").

In [13]:
def sum_good_continue(nums: List[int], k: int) -> int:
    n = len(nums)
    ans = 0
    for i, x in enumerate(nums):
        if i >= k and x <= nums[i - k]:        # có hàng xóm trái mà x không lớn hơn hẳn -> loại
            continue
        if i + k < n and x <= nums[i + k]:     # có hàng xóm phải mà x không lớn hơn hẳn -> loại
            continue
        ans += x
    return ans


for nums, k, expected in examples:
    print(sum_good_continue(nums, k), "(kỳ vọng", expected, ")")

12 (kỳ vọng 12 )
2 (kỳ vọng 2 )


In [14]:
# Code nộp LeetCode cho Cách 3
class SolutionContinue:
    def sumOfGoodNumbers(self, nums: List[int], k: int) -> int:
        n = len(nums)
        ans = 0
        for i, x in enumerate(nums):
            if i >= k and x <= nums[i - k]:
                continue
            if i + k < n and x <= nums[i + k]:
                continue
            ans += x
        return ans


print(SolutionContinue().sumOfGoodNumbers([2, 1], 1))      # 2

2


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 4 — Một biểu thức `sum(...)` với điều kiện gộp

**Giải thích từng đoạn.**

1. Điều kiện **tốt** = "(không có hàng xóm trái **hoặc** `x` lớn hơn hẳn nó) **và** (không có hàng xóm phải **hoặc** `x` lớn hơn hẳn nó)".
2. "Không có hàng xóm trái" viết là `i < k`; "không có hàng xóm phải" viết là `i + k >= n`.
3. Nhờ đoản mạch của `or`, `nums[i - k]` chỉ được tính khi `i >= k` — không lấy nhầm chỉ số âm.

In [15]:
def sum_good_oneliner(nums: List[int], k: int) -> int:
    n = len(nums)
    return sum(
        x
        for i, x in enumerate(nums)
        if (i < k or x > nums[i - k]) and (i + k >= n or x > nums[i + k])
    )


for nums, k, expected in examples:
    print(sum_good_oneliner(nums, k), "(kỳ vọng", expected, ")")

12 (kỳ vọng 12 )
2 (kỳ vọng 2 )


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(1)`**. Ngắn gọn, nhưng người mới dễ đọc nhầm `and`/`or`; hãy đọc từng vế như phần giải thích ở trên.

### Cách 5 — Đệm mảng hai đầu bằng `0`

**Giải thích từng đoạn.**

1. `padded = [0] * k + nums + [0] * k`: thêm `k` số `0` mỗi đầu. Phần tử `nums[i]` nay nằm ở `padded[i + k]`.
2. Hàng xóm trái `nums[i - k]` ↔ `padded[i]`; hàng xóm phải `nums[i + k]` ↔ `padded[i + 2k]`. Cả hai **luôn là chỉ số hợp lệ** (`0 <= i` và `i + 2k <= n - 1 + 2k`).
3. Phần tử đệm `0` nhỏ hơn mọi `nums[i] >= 1` nên không cản.
4. Không cần kiểm tra biên ở vòng lặp — chỉ so sánh.

In [16]:
def sum_good_padding(nums: List[int], k: int) -> int:
    padded = [0] * k + nums + [0] * k            # đệm k số 0 mỗi đầu
    total = 0
    for i in range(len(nums)):
        x = padded[i + k]                        # chính nums[i]
        if x > padded[i] and x > padded[i + 2 * k]:
            total += x
    return total


for nums, k, expected in examples:
    print(sum_good_padding(nums, k), "(kỳ vọng", expected, ")")

12 (kỳ vọng 12 )
2 (kỳ vọng 2 )


**Độ phức tạp Cách 5:** **Time `O(n + k)`** = `O(n)` (vì `k <= n / 2`), **Space `O(n + k)`** = `O(n)` (mảng đệm). Tốn thêm bộ nhớ nhưng loại bỏ hết kiểm tra biên.

## 8. TRACE từng bước

Ta trace Cách 3 (`continue`) trên `nums = [1, 3, 2, 1, 5, 4]`, `k = 2` và trên `nums = [2, 1]`, `k = 1`.

In [17]:
def trace_continue(nums, k):
    n = len(nums)
    print(f"nums = {nums}, k = {k}, n = {n}")
    ans = 0
    for i, x in enumerate(nums):
        if i >= k and x <= nums[i - k]:
            print(f"  i={i} x={x}: trái nums[{i - k}]={nums[i - k]}, {x} <= {nums[i - k]} -> LOẠI")
            continue
        if i + k < n and x <= nums[i + k]:
            print(f"  i={i} x={x}: phải nums[{i + k}]={nums[i + k]}, {x} <= {nums[i + k]} -> LOẠI")
            continue
        ans += x
        left = nums[i - k] if i >= k else "không có"
        right = nums[i + k] if i + k < n else "không có"
        print(f"  i={i} x={x}: trái={left}, phải={right} -> TỐT, ans = {ans}")
    return ans


assert trace_continue([1, 3, 2, 1, 5, 4], 2) == 12
assert trace_continue([2, 1], 1) == 2

nums = [1, 3, 2, 1, 5, 4], k = 2, n = 6
  i=0 x=1: phải nums[2]=2, 1 <= 2 -> LOẠI
  i=1 x=3: trái=không có, phải=1 -> TỐT, ans = 3
  i=2 x=2: phải nums[4]=5, 2 <= 5 -> LOẠI
  i=3 x=1: trái nums[1]=3, 1 <= 3 -> LOẠI
  i=4 x=5: trái=2, phải=không có -> TỐT, ans = 8
  i=5 x=4: trái=1, phải=không có -> TỐT, ans = 12
nums = [2, 1], k = 1, n = 2
  i=0 x=2: trái=không có, phải=1 -> TỐT, ans = 2
  i=1 x=1: trái nums[0]=2, 1 <= 2 -> LOẠI


In [18]:
def trace_padding(nums, k):
    padded = [0] * k + nums + [0] * k
    print(f"nums = {nums}, k = {k}, padded = {padded}")
    total = 0
    for i in range(len(nums)):
        x, left, right = padded[i + k], padded[i], padded[i + 2 * k]
        good = x > left and x > right
        if good:
            total += x
        print(f"  i={i}: x={x}, trái={left}, phải={right} -> {'TỐT' if good else 'không'}, tổng = {total}")
    return total


assert trace_padding([1, 3, 2, 1, 5, 4], 2) == 12

nums = [1, 3, 2, 1, 5, 4], k = 2, padded = [0, 0, 1, 3, 2, 1, 5, 4, 0, 0]
  i=0: x=1, trái=0, phải=2 -> không, tổng = 0
  i=1: x=3, trái=0, phải=1 -> TỐT, tổng = 3
  i=2: x=2, trái=1, phải=5 -> không, tổng = 3
  i=3: x=1, trái=3, phải=4 -> không, tổng = 3
  i=4: x=5, trái=2, phải=0 -> TỐT, tổng = 8
  i=5: x=4, trái=1, phải=0 -> TỐT, tổng = 12


Bảng trace Cách 3 với `nums = [1, 3, 2, 1, 5, 4]`, `k = 2`:

| `i` | `x` | hàng xóm trái | hàng xóm phải | kết quả | `ans` |
|---|---|---|---|---|---|
| 0 | 1 | không có | `nums[2] = 2` | `1 <= 2` loại | 0 |
| 1 | 3 | không có | `nums[3] = 1` | `3 > 1` tốt | 3 |
| 2 | 2 | `nums[0] = 1` (qua) | `nums[4] = 5` | `2 <= 5` loại | 3 |
| 3 | 1 | `nums[1] = 3` | — | `1 <= 3` loại ở bước trái | 3 |
| 4 | 5 | `nums[2] = 2` (qua) | không có | tốt | 8 |
| 5 | 4 | `nums[3] = 1` (qua) | không có | tốt | 12 |

Kết quả `12`.

## 9. Code nộp LeetCode

Mình nộp Cách 3 (`continue`): đọc đúng như đề, `O(n)` thời gian, `O(1)` bộ nhớ. Trên LeetCode `List` có sẵn; để chạy ở máy bạn, ghi rõ `from typing import List`.

In [19]:
from typing import List


class Solution:
    def sumOfGoodNumbers(self, nums: List[int], k: int) -> int:
        n = len(nums)
        ans = 0
        for i, x in enumerate(nums):
            if i >= k and x <= nums[i - k]:         # có hàng xóm trái nhưng x không lớn hơn hẳn
                continue
            if i + k < n and x <= nums[i + k]:      # có hàng xóm phải nhưng x không lớn hơn hẳn
                continue
            ans += x
        return ans


sol = Solution()
for nums, k, expected in examples:
    print(nums, k, "->", sol.sumOfGoodNumbers(nums, k), "(kỳ vọng", expected, ")")

[1, 3, 2, 1, 5, 4] 2 -> 12 (kỳ vọng 12 )
[2, 1] 1 -> 2 (kỳ vọng 2 )


## 10. Vì sao thuật toán đúng

- **Đúng định nghĩa "tốt":** `nums[i]` tốt ⇔ với mỗi chỉ số trong `{i - k, i + k}` **nằm trong mảng**, `nums[i] > nums[chỉ số đó]`. Phủ định: `nums[i]` **không** tốt ⇔ tồn tại một chỉ số tồn tại mà `nums[i] <= nums[chỉ số đó]`. Hai lệnh `if ... continue` kiểm tra đúng phủ định này cho hai chỉ số.
- **Biên an toàn:** `i >= k` đảm bảo `i - k >= 0` trước khi truy cập; `i + k < n` đảm bảo `i + k <= n - 1`. Nhờ đoản mạch của `and`, `nums[i - k]` / `nums[i + k]` không bao giờ bị tính khi chỉ số không hợp lệ.
- **Trường hợp không có hàng xóm nào:** nếu cả hai chỉ số ngoài mảng thì không `continue` nào chạy ⇒ `x` được cộng — đúng với câu "vẫn được coi là tốt" của đề.
- **Hòa không tốt:** ta dùng `x <= hàng_xóm` để loại, tức `x > hàng_xóm` mới tốt ("strictly greater").
- **Đủ và không trùng:** mỗi `i` được xét đúng một lần và cộng tối đa một lần ⇒ tổng đúng.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. Cờ `good` | `O(n)` | `O(1)` |
| 2. Hàm lính canh | `O(n)` | `O(1)` |
| 3. `continue` | `O(n)` | `O(1)` |
| 4. Một biểu thức `sum` | `O(n)` | `O(1)` |
| 5. Đệm mảng | `O(n)` | `O(n)` |

Mỗi phần tử chỉ so với tối đa hai phần tử khác nên không thể tốt hơn `O(n)` (phải đọc cả mảng). Với `n <= 100`, mọi cách chạy tức thì.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng bằng `assert`.

In [20]:
sol = Solution()
impls = {
    "flag": sum_good_flag,
    "sentinel": sum_good_sentinel,
    "continue": sum_good_continue,
    "oneliner": sum_good_oneliner,
    "padding": sum_good_padding,
    "leetcode": sol.sumOfGoodNumbers,
}


def check_all(nums, k, expected, label):
    for key, fn in impls.items():
        actual = fn(list(nums), k)
        assert actual == expected, (label, key, nums, k, actual, expected)
    print("PASS", label)


for idx, (nums, k, expected) in enumerate(examples, 1):
    check_all(nums, k, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2


### 12.1. Các trường hợp biên

Các mảng bên dưới là **tự dựng** (kỳ vọng tính tay, ghi trong nhãn); đều thỏa ràng buộc `2 <= n <= 100`, `1 <= nums[i] <= 1000`, `1 <= k <= n // 2`.

In [21]:
check_all([5, 5], 1, 0, "hai số bằng nhau: không ai lớn hơn hẳn")
check_all([1, 2], 1, 2, "tăng dần, chỉ số 1 thắng chỉ số 0")
check_all([3, 2, 1], 1, 3, "giảm dần: i=0 tốt (3>2); i=1: 2>1 nhưng 2<3; i=2: 1<2")
check_all([1, 2, 3], 1, 3, "tăng dần: chỉ phần tử cuối tốt")
check_all([2, 2, 2, 2], 2, 0, "toàn bằng nhau, k = n // 2")
check_all([1, 2, 3, 4], 2, 7, "k = 2: i=2 (3>1) và i=3 (4>2) tốt; i=0: 1<3; i=1: 2<4")
check_all([1000, 1, 1000], 1, 2000, "k = 1: i=0 (1000>1) tốt, i=2 (1000>1) tốt, i=1 (1) xấu")
check_all([4, 7, 2, 8, 6], 2, 4 + 8 + 6, "i=0: 4>2 tốt; i=1: 7>8 sai; i=2: 2>4 sai; i=3: 8>7 tốt; i=4: 6>2 tốt")
check_all([1] * 100, 50, 0, "n = 100 toàn số 1, k = 50: không ai lớn hơn hẳn")
check_all(list(range(1, 101)), 50, sum(range(51, 101)), "tăng dần 1..100, k=50: chỉ i>=50 tốt (i<50 thua hàng xóm phải)")
check_all(list(range(100, 0, -1)), 50, sum(range(100, 50, -1)), "giảm dần 100..1, k=50: chỉ i<50 tốt")

PASS hai số bằng nhau: không ai lớn hơn hẳn
PASS tăng dần, chỉ số 1 thắng chỉ số 0
PASS giảm dần: i=0 tốt (3>2); i=1: 2>1 nhưng 2<3; i=2: 1<2
PASS tăng dần: chỉ phần tử cuối tốt
PASS toàn bằng nhau, k = n // 2
PASS k = 2: i=2 (3>1) và i=3 (4>2) tốt; i=0: 1<3; i=1: 2<4
PASS k = 1: i=0 (1000>1) tốt, i=2 (1000>1) tốt, i=1 (1) xấu
PASS i=0: 4>2 tốt; i=1: 7>8 sai; i=2: 2>4 sai; i=3: 8>7 tốt; i=4: 6>2 tốt
PASS n = 100 toàn số 1, k = 50: không ai lớn hơn hẳn
PASS tăng dần 1..100, k=50: chỉ i>=50 tốt (i<50 thua hàng xóm phải)
PASS giảm dần 100..1, k=50: chỉ i<50 tốt


### 12.2. Phản ví dụ cho các cách làm sai

Hai cách sai điển hình: (a) **không kiểm tra biên trái** (chỉ số âm quay vòng); (b) dùng `>=` thay `>` (cho phép hòa).

In [22]:
def wrong_negative_index(nums, k):
    n = len(nums)
    total = 0
    for i, x in enumerate(nums):
        if x > nums[i - k] and (i + k >= n or x > nums[i + k]):     # SAI: nums[i - k] với i < k lấy phần tử cuối mảng
            total += x
    return total


def wrong_ge(nums, k):
    n = len(nums)
    total = 0
    for i, x in enumerate(nums):
        if (i < k or x >= nums[i - k]) and (i + k >= n or x >= nums[i + k]):   # SAI: dùng >=
            total += x
    return total


print(wrong_negative_index([1, 3, 2, 1, 5, 4], 2), "!=", sum_good_continue([1, 3, 2, 1, 5, 4], 2))
print(wrong_ge([5, 5], 1), "!=", sum_good_continue([5, 5], 1))
assert wrong_negative_index([1, 3, 2, 1, 5, 4], 2) != 12
assert wrong_ge([5, 5], 1) != 0

9 != 12
10 != 0


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh mảng ngẫu nhiên **đúng ràng buộc**: `n` trong `2..100`, `nums[i]` trong `1..1000`, `k` trong `1..n // 2`. Để có nhiều ca hòa, ta dùng cả khoảng giá trị nhỏ (`1..3`). Ngoài năm cách, ta còn đối chiếu với một **định nghĩa viết hoàn toàn độc lập**: dựng danh sách hàng xóm rồi dùng `all`.

In [23]:
def reference(nums, k):
    n = len(nums)
    total = 0
    for i in range(n):
        neighbours = [nums[j] for j in (i - k, i + k) if 0 <= j < n]     # hàng xóm tồn tại
        if all(nums[i] > v for v in neighbours):                          # all([]) = True: không hàng xóm vẫn tốt
            total += nums[i]
    return total


def valid(nums, k):
    n = len(nums)
    return 2 <= n <= 100 and all(1 <= v <= 1000 for v in nums) and 1 <= k <= n // 2


rng = random.Random(3452)
for trial in range(3000):
    n = rng.randint(2, 100)
    hi = rng.choice([3, 10, 1000])
    nums = [rng.randint(1, hi) for _ in range(n)]
    k = rng.randint(1, n // 2)
    assert valid(nums, k)
    ref = reference(nums, k)
    for key, fn in impls.items():
        assert fn(list(nums), k) == ref, (key, nums, k)
print("PASS 3000 mảng ngẫu nhiên đúng ràng buộc")

PASS 3000 mảng ngẫu nhiên đúng ràng buộc


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Không kiểm tra biên trái — chỉ số âm lặng lẽ quay vòng

In [24]:
def sai_chi_so_am(nums, k):
    total = 0
    for i, x in enumerate(nums):
        if x > nums[i - k] and (i + k >= len(nums) or x > nums[i + k]):   # SAI: i - k có thể âm
            total += x
    return total


print("sai:", sai_chi_so_am([1, 3, 2, 1, 5, 4], 2), "| đúng:", Solution().sumOfGoodNumbers([1, 3, 2, 1, 5, 4], 2))
print("i=1, k=2 -> nums[i - k] = nums[-1] =", [1, 3, 2, 1, 5, 4][1 - 2], "(phần tử cuối, không phải 'không tồn tại')")

sai: 9 | đúng: 12
i=1, k=2 -> nums[i - k] = nums[-1] = 4 (phần tử cuối, không phải 'không tồn tại')


**Vì sao sai:** với `i = 1`, `k = 2`, `i - k = -1`, và `nums[-1]` là phần tử **cuối** (`4`). Số `3` bị so với `4` (và thua) dù theo đề hàng xóm trái **không tồn tại**. Python không báo lỗi nên rất khó phát hiện. **Cách sửa:** kiểm tra `i >= k` trước (hoặc `i - k >= 0`), dùng `and` đoản mạch hoặc đệm mảng.

### Lỗi 2: Dùng `>=` thay vì `>` (cho phép hòa)

In [25]:
def sai_dau_bang(nums, k):
    n = len(nums)
    total = 0
    for i, x in enumerate(nums):
        if i >= k and x < nums[i - k]:           # SAI: chỉ loại khi nhỏ hơn hẳn, hòa vẫn cho qua
            continue
        if i + k < n and x < nums[i + k]:
            continue
        total += x
    return total


print("sai:", sai_dau_bang([5, 5], 1), "| đúng:", Solution().sumOfGoodNumbers([5, 5], 1))

sai: 10 | đúng: 0


**Vì sao sai:** đề nói **strictly greater** (lớn hơn hẳn), nên hai số **bằng nhau** thì phần tử **không** tốt. Với `[5, 5]`, `k = 1`: mỗi số hòa với số kia nên đáp án là `0`, nhưng bản sai cho `10`. **Cách sửa:** loại khi `x <= hàng_xóm` (hoặc tốt khi `x > hàng_xóm`).

### Lỗi 3: So với `i - 1` / `i + 1` thay vì `i - k` / `i + k`

In [26]:
def sai_hang_xom_ke(nums, k):
    n = len(nums)
    total = 0
    for i, x in enumerate(nums):
        if i >= 1 and x <= nums[i - 1]:          # SAI: đề so với i - k, không phải i - 1
            continue
        if i + 1 < n and x <= nums[i + 1]:
            continue
        total += x
    return total


print("sai:", sai_hang_xom_ke([1, 3, 2, 1, 5, 4], 2), "| đúng:", Solution().sumOfGoodNumbers([1, 3, 2, 1, 5, 4], 2))

sai: 8 | đúng: 12


**Vì sao sai:** `k` là **khoảng cách** giữa hai chỉ số cần so sánh, không phải luôn `1`. Ở ví dụ 1 (`k = 2`) đáp án đúng `12` nhưng bản sai cho kết quả khác. **Cách sửa:** luôn dùng biến `k` của đề.

### Lỗi 4: Kiểm tra biên phải sai (`i + k <= n`)

In [27]:
def sai_bien_phai(nums, k):
    n = len(nums)
    total = 0
    for i, x in enumerate(nums):
        if i >= k and x <= nums[i - k]:
            continue
        if i + k <= n and x <= nums[i + k]:      # SAI: phải là i + k < n (chỉ số tối đa n - 1)
            continue
        total += x
    return total


try:
    sai_bien_phai([1, 2], 1)
except IndexError as e:
    print("IndexError:", e)
print("đúng:", Solution().sumOfGoodNumbers([1, 2], 1))

IndexError: list index out of range
đúng: 2


**Vì sao sai:** chỉ số hợp lệ lớn nhất là `n - 1`, nên điều kiện đúng là `i + k < n`. Với `<=`, khi `i + k == n` (ví dụ `[1, 2]`, `k = 1`, `i = 1`) ta truy cập `nums[n]` và gặp `IndexError`. **Cách sửa:** `i + k < n` (hoặc `i + k <= n - 1`).

### Lỗi 5: Đếm số phần tử tốt thay vì cộng giá trị

In [28]:
def sai_dem_so_luong(nums, k):
    n = len(nums)
    count = 0
    for i, x in enumerate(nums):
        if i >= k and x <= nums[i - k]:
            continue
        if i + k < n and x <= nums[i + k]:
            continue
        count += 1                                # SAI: đề yêu cầu TỔNG GIÁ TRỊ
    return count


print("sai:", sai_dem_so_luong([1, 3, 2, 1, 5, 4], 2), "| đúng:", Solution().sumOfGoodNumbers([1, 3, 2, 1, 5, 4], 2))

sai: 3 | đúng: 12


**Vì sao sai:** ba phần tử tốt `3, 5, 4` có **số lượng** `3` nhưng **tổng** `12`. **Cách sửa:** `ans += x` (cộng giá trị, không cộng `1`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "So `nums[i]` với phần tử **cách `k` bước**" ⇒ duyệt `enumerate`, xét `i - k` và `i + k` với **kiểm tra biên**.
- Python: **chỉ số âm không báo lỗi** — luôn kiểm tra `i - k >= 0` **trước** khi đọc `nums[i - k]`.
- Cặp điều kiện biên cần nhớ: trái `i >= k`, phải `i + k < n`.
- Điều kiện kiểu "thắng mọi hàng xóm tồn tại" ⇒ cách viết gọn: `all(x > v for v in neighbours)`; `all` của danh sách rỗng là `True`, khớp với "không có hàng xóm vẫn tốt".
- Khi muốn tránh kiểm tra biên: **đệm** hai đầu bằng giá trị không cản (ở đây `0` vì `nums[i] >= 1`; tổng quát dùng `-inf`).
- "Strictly" ⇒ `>` hoặc `<`; "at least" ⇒ `>=`/`<=`. Đọc kỹ từng chữ.
- Kiểm tra nhanh: ví dụ 2 (`[2, 1]`, `k = 1`) có biên trái **và** phải cùng lúc.

## 15. Biến thể của bài

1. **Đếm** số phần tử tốt (thay vì tổng).
2. **Liệt kê chỉ số** của các phần tử tốt.
3. **Tốt nếu lớn hơn mọi phần tử trong cửa sổ bán kính `k`** (`nums[i - k .. i + k]`) — gần với cực đại cửa sổ trượt.
4. **Không nghiêm ngặt:** tốt nếu `>=` hàng xóm; hoặc tốt nếu lớn hơn hẳn **ít nhất một** hàng xóm.
5. Bài cùng họ: **Find Peak Element (162)**, **Peak Index in a Mountain Array (852)**, **Find All Lonely Numbers in the Array (2150)**.

In [29]:
def good_indices(nums, k):
    n = len(nums)
    return [i for i, x in enumerate(nums)
            if (i < k or x > nums[i - k]) and (i + k >= n or x > nums[i + k])]


def count_good(nums, k):
    return len(good_indices(nums, k))


def sum_good_ge(nums, k):
    """Biến thể không nghiêm ngặt: tốt nếu >= các hàng xóm tồn tại."""
    n = len(nums)
    return sum(x for i, x in enumerate(nums)
               if (i < k or x >= nums[i - k]) and (i + k >= n or x >= nums[i + k]))


def sum_good_window(nums, k):
    """Biến thể: tốt nếu lớn hơn hẳn MỌI phần tử khác trong cửa sổ [i - k, i + k]."""
    n = len(nums)
    total = 0
    for i, x in enumerate(nums):
        window = [nums[j] for j in range(max(0, i - k), min(n, i + k + 1)) if j != i]
        if all(x > v for v in window):
            total += x
    return total


nums, k = [1, 3, 2, 1, 5, 4], 2
print(good_indices(nums, k), count_good(nums, k), sum_good_ge([5, 5], 1), sum_good_window(nums, k))
assert good_indices(nums, k) == [1, 4, 5]
assert count_good(nums, k) == 3
assert sum_good_ge([5, 5], 1) == 10
assert sum_good_window([1, 3, 2, 1, 5, 4], 1) == 3 + 5      # cửa sổ bán kính 1: 3 > 1,2 ; 5 > 1,4 ; 4 > 5? sai

[1, 4, 5] 3 10 8


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Cờ `good` | hai khối `if` đặt `good = False` | `O(n)` | `O(1)` | dễ kiểm tra từng điều kiện |
| 2. Hàm lính canh | ngoài mảng coi như `0` | `O(n)` | `O(1)` | khi muốn bỏ kiểm tra biên ở vòng lặp chính |
| 3. `continue` | loại sớm nếu thua một hàng xóm | `O(n)` | `O(1)` | **nên nộp**, đọc đúng như đề |
| 4. Một biểu thức `sum` | gộp điều kiện bằng `and`/`or` | `O(n)` | `O(1)` | ngắn gọn, khi đã quen đoản mạch |
| 5. Đệm mảng | thêm `k` số `0` mỗi đầu | `O(n)` | `O(n)` | muốn vòng lặp không có biên |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `[1, 2, 3][-1]`, `3 > 3`, `sum([])`, `Solution().sumOfGoodNumbers([3, 1], 1)`.

**Bài 2 (tính tay).** Với `nums = [4, 1, 6, 2, 5]` và `k = 1`: viết hàng xóm trái, hàng xóm phải của từng vị trí, xác định số tốt và tổng.

**Bài 3 (điền TODO).** Hoàn thành `sum_good_todo(nums, k)` theo Cách 3 (`continue`).

In [30]:
def sum_good_todo(nums, k):
    # TODO 1: n = len(nums), ans = 0
    # TODO 2: for i, x in enumerate(nums): nếu có hàng xóm trái (i >= k) và x <= nums[i - k] -> continue
    # TODO 3: nếu có hàng xóm phải (i + k < n) và x <= nums[i + k] -> continue
    # TODO 4: ans += x ; cuối cùng return ans
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `sum_good_padding2(nums, k)` theo Cách 5 nhưng dùng **một vòng `for i, x in enumerate(nums)`** và truy cập `padded[i]`, `padded[i + 2 * k]` (không dùng `range(len(nums))`).

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_good(nums, k)` đếm số phần tử tốt. (b) Viết `max_good(nums, k)` trả về **phần tử tốt lớn nhất**, hoặc `0` nếu không có phần tử tốt nào (hãy nghĩ ví dụ mảng không có phần tử tốt).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `3`, `False`, `0`, `3`. Bài 3: ba mệnh đề: trái (`i >= k`), phải (`i + k < n`), rồi `ans += x`.
- **Mức 2 (vừa):** Bài 2 (`k = 1`): `i=0`: trái không có, phải `1`, `4 > 1` tốt; `i=1`: `1` thua `4` ⇒ loại; `i=2`: `6 > 1` và `6 > 2` tốt; `i=3`: `2` thua `6` ⇒ loại; `i=4`: trái `2`, `5 > 2` tốt. Tổng `4 + 6 + 5 = 15`. Bài 4: `padded = [0]*k + nums + [0]*k`, so `x` với `padded[i]` và `padded[i + 2*k]`.
- **Mức 3 (sâu):** Bài 5(a): tương tự Cách 3 nhưng `count += 1`. 5(b): có thể **không** có phần tử tốt nào (ví dụ `[5, 5]`, `k = 1`), nên gom các phần tử tốt vào list rồi dùng `max(..., default=0)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert [1, 2, 3][-1] == 3
assert (3 > 3) is False
assert sum([]) == 0
assert Solution().sumOfGoodNumbers([3, 1], 1) == 3

# Bài 2
assert Solution().sumOfGoodNumbers([4, 1, 6, 2, 5], 1) == 4 + 6 + 5


# Bài 3
def sum_good_todo_ans(nums, k):
    n = len(nums)
    ans = 0
    for i, x in enumerate(nums):
        if i >= k and x <= nums[i - k]:
            continue
        if i + k < n and x <= nums[i + k]:
            continue
        ans += x
    return ans


for nums, k, expected in examples:
    assert sum_good_todo_ans(nums, k) == expected


# Bài 4
def sum_good_padding2(nums, k):
    padded = [0] * k + nums + [0] * k
    total = 0
    for i, x in enumerate(nums):
        if x > padded[i] and x > padded[i + 2 * k]:
            total += x
    return total


for nums, k, expected in examples:
    assert sum_good_padding2(nums, k) == expected
assert sum_good_padding2([4, 1, 6, 2, 5], 1) == 15


# Bài 5(a)
def count_good_ans(nums, k):
    n = len(nums)
    count = 0
    for i, x in enumerate(nums):
        if i >= k and x <= nums[i - k]:
            continue
        if i + k < n and x <= nums[i + k]:
            continue
        count += 1
    return count


assert count_good_ans([1, 3, 2, 1, 5, 4], 2) == 3
assert count_good_ans([2, 1], 1) == 1
assert count_good_ans([5, 5], 1) == 0


# Bài 5(b)
def max_good(nums, k):
    n = len(nums)
    good = [x for i, x in enumerate(nums)
            if (i < k or x > nums[i - k]) and (i + k >= n or x > nums[i + k])]
    return max(good, default=0)           # có thể không có phần tử tốt nào, ví dụ [5, 5] với k = 1


assert max_good([1, 3, 2, 1, 5, 4], 2) == 5
assert max_good([2, 1], 1) == 2
assert max_good([5, 5], 1) == 0
print("Tất cả đáp án bài tập đều đúng")

Tất cả đáp án bài tập đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (duyệt `i, x`; loại nếu thua/hòa một hàng xóm **tồn tại**; cộng giá trị còn lại)

```python
from typing import List


class Solution:
    def sumOfGoodNumbers(self, nums: List[int], k: int) -> int:
        n = len(nums)
        ans = 0
        for i, x in enumerate(nums):
            if i >= k and x <= nums[i - k]:
                continue
            if i + k < n and x <= nums[i + k]:
                continue
            ans += x
        return ans
```

**Checklist**

- [ ] Kiểm tra biên **trước** khi đọc `nums[i - k]` (`i >= k`) và `nums[i + k]` (`i + k < n`)?
- [ ] Dùng so sánh **nghiêm ngặt**: loại khi `x <= hàng_xóm` (hòa cũng bị loại)?
- [ ] Dùng khoảng cách `k` (không phải `1`)?
- [ ] Cộng **giá trị** `x` (không phải đếm `1`)?
- [ ] Đã thử `[1,3,2,1,5,4]`, `k=2` (`12`), `[2,1]`, `k=1` (`2`) và `[5,5]`, `k=1` (`0`)?
- [ ] Skeleton đúng `class Solution` / `def sumOfGoodNumbers(self, nums: List[int], k: int) -> int`?